# Non-AI Solution: Screw and Bolt Detection using OpenCV
**Author:** Sanjay  
**Task:** Count items in images using traditional Computer Vision techniques  
**Target Accuracy:** > 95%

## Approach
- Binary thresholding for object segmentation
- Contour detection for object identification
- Shape analysis (hexagonal/circular) for bolt detection
- Center hole detection for screw vs bolt classification

In [ ]:
# Install required packages
!pip install opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path

# For Colab file upload
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
class ScrewBoltDetector:
    """Advanced detector for screws and bolts using traditional CV techniques."""
    
    def __init__(self, min_area=100, max_area=10000, blur_kernel=(5, 5), threshold=60):
        self.min_area = min_area
        self.max_area = max_area
        self.blur_kernel = blur_kernel
        self.threshold = threshold
    
    def is_hexagonal_or_circular(self, contour):
        """Detect hexagonal (bolt head) or circular shapes."""
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        # Hexagonal detection
        if 5 <= len(approx) <= 7:
            return True
        
        # Circular detection
        if len(approx) > 7:
            area = cv2.contourArea(contour)
            perimeter = cv2.arcLength(contour, True)
            circularity = 4 * np.pi * area / (perimeter ** 2 + 1e-5)
            return circularity > 0.6
        
        return False
    
    def has_center_hole(self, roi_image):
        """Detect if ROI has a center hole (indicates bolt)."""
        roi_gray = cv2.cvtColor(roi_image, cv2.COLOR_BGR2GRAY)
        roi_blur = cv2.GaussianBlur(roi_gray, (3, 3), 0)
        _, roi_thresh = cv2.threshold(roi_blur, self.threshold, 255, cv2.THRESH_BINARY_INV)
        
        inner_contours, _ = cv2.findContours(
            roi_thresh, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE
        )
        
        for inner_cnt in inner_contours:
            area = cv2.contourArea(inner_cnt)
            if 20 < area < 1000:
                if self.is_hexagonal_or_circular(inner_cnt):
                    return True
        
        return False
    
    def detect_objects(self, image):
        """Detect and classify screws and bolts in image."""
        output = image.copy()
        
        # Preprocessing
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, self.blur_kernel, 0)
        _, thresh = cv2.threshold(blurred, self.threshold, 255, cv2.THRESH_BINARY_INV)
        
        # Morphological operations
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
        
        # Find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        screw_count = 0
        bolt_count = 0
        detections = []
        
        for cnt in contours:
            area = cv2.contourArea(cnt)
            
            if area < self.min_area or area > self.max_area:
                continue
            
            x, y, w, h = cv2.boundingRect(cnt)
            roi = image[y:y+h, x:x+w]
            
            # Classify
            if self.has_center_hole(roi):
                label = 'Bolt'
                bolt_count += 1
                color = (0, 0, 255)  # Red
            else:
                label = 'Screw'
                screw_count += 1
                color = (0, 255, 0)  # Green
            
            detections.append({
                'type': label,
                'bbox': (x, y, w, h),
                'area': int(area)
            })
            
            # Draw annotations
            cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)
            cv2.putText(output, label, (x, y - 10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        return output, screw_count, bolt_count, detections

In [ ]:
# Upload images (for Colab)
if IN_COLAB:
    uploaded = files.upload()
    image_files = list(uploaded.keys())
else:
    # Automatically load all images from the specified directory
    import glob
    image_dir = '/Users/sanju/Desktop/Sanjay/Screws_2024_07_15'
    image_files = sorted(
        glob.glob(f"{image_dir}/*.jpg") +
        glob.glob(f"{image_dir}/*.jpeg") +
        glob.glob(f"{image_dir}/*.png") +
        glob.glob(f"{image_dir}/*.bmp")
    )
    print(f"Found {len(image_files)} images in {image_dir}")


In [ ]:
# Initialize detector
detector = ScrewBoltDetector(
    min_area=100,
    max_area=10000,
    threshold=60
)

# Process each image
results = []

for fname in image_files:
    image = cv2.imread(fname)
    
    if image is None:
        print(f"Could not read {fname}")
        continue
    
    annotated, screws, bolts, detections = detector.detect_objects(image)
    total = screws + bolts
    
    results.append({
        'filename': fname,
        'screws': screws,
        'bolts': bolts,
        'total': total
    })
    
    print(f"{fname} -> Screws: {screws}, Bolts: {bolts}, Total: {total}")
    
    # Display results
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f"{fname} - Screws: {screws}, Bolts: {bolts}, Total: {total}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Summary statistics
import pandas as pd

df = pd.DataFrame(results)
print("\n=== Detection Summary ===")
print(df)
print(f"\nTotal objects detected: {df['total'].sum()}")
print(f"Total screws: {df['screws'].sum()}")
print(f"Total bolts: {df['bolts'].sum()}")

## Key Techniques Used

1. **Preprocessing**: Grayscale conversion, Gaussian blur, binary thresholding
2. **Morphological Operations**: Close and open operations to reduce noise
3. **Contour Detection**: RETR_EXTERNAL for object boundaries
4. **Shape Analysis**: Polygon approximation for hexagon detection
5. **Circularity Metric**: Area/perimeter ratio for circular shapes
6. **ROI Analysis**: Center hole detection for bolt classification

## Advantages
- Fast inference (no model loading)
- Lightweight (no GPU required)
- Interpretable logic
- Easy to tune parameters

## Limitations
- Sensitive to lighting conditions
- Requires parameter tuning per dataset
- May struggle with overlapping objects